# Session 13: Grouping and Combining Tables

Data Science with Python, Level 1, Stage 2: Working with Data

## Learning Objectives

By the end of this session you will be able to:

1. Group rows and calculate a statistic per group.
2. Build summary tables with agg and pivot_table.
3. Group by several columns and by parts of dates.
4. Join tables on a key with merge.
5. Choose between inner and left joins and check the result.
6. Stack tables with concat and reshape with melt.

## Where We Are

In Session 11 you compared the three cafe branches by filtering one branch at a time. That works for three branches, but not for 40 products or 365 days. **Grouping** answers the same question for every group in one line: split the rows into groups, calculate, and combine the results into a table.

Real systems also store data in **several tables** that refer to each other: customers in one, products in another, orders in a third, linked by ID columns. In the second half you will **join** tables to answer questions that no single table can answer.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]
df["date"] = pd.to_datetime(df["date"])

customers = pd.read_csv(DATA_URL + "shop_customers.csv")
products = pd.read_csv(DATA_URL + "shop_products.csv")
orders = pd.read_csv(DATA_URL + "shop_orders.csv")

## 1. Split, Apply, Combine

In [ ]:
df.groupby("branch")["revenue"].sum().round(2)

In [ ]:
print(df.groupby("category")["revenue"].mean().round(2))
print()
print(df.groupby("branch")["order_id"].count())

## 2. Several Statistics with agg

In [ ]:
summary = df.groupby("branch").agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
    items=("quantity", "sum"),
)
summary.round(2).sort_values("revenue", ascending=False)

| Name | Meaning |
| --- | --- |
| `"count"`, `"size"` | Number of values, or of rows |
| `"sum"`, `"mean"`, `"median"` | Total, average, middle value |
| `"min"`, `"max"` | Smallest and largest |
| `"std"` | Standard deviation (spread) |
| `"nunique"` | Number of different values |
| `"first"`, `"last"` | First or last value in each group |

## 3. Grouping by Several Columns

In [ ]:
by_two = df.groupby(["branch", "category"])["revenue"].sum().round(1)
print(by_two.head(7))

In [ ]:
monthly = df.groupby(df["date"].dt.month)["revenue"].sum().round(2)
print(monthly)

hourly = df.groupby("hour")["order_id"].count()
print("Busiest hour:", hourly.idxmax(), "with", hourly.max(), "orders")

## 4. Pivot Tables and Cross Tables

In [ ]:
pd.pivot_table(df, index="category", columns="branch",
               values="revenue", aggfunc="sum", margins=True).round(1)

## 5. Group Results on Every Row

`agg` returns one row per group. Sometimes you want the group result next to **every original row**, for example to compare each order with its branch average. `transform` does exactly that: it returns a Series as long as the original table.

In [ ]:
df["branch_avg"] = df.groupby("branch")["revenue"].transform("mean")
df["above_avg"] = df["revenue"] > df["branch_avg"]

print(df[["branch", "revenue", "branch_avg", "above_avg"]].head(4).round(2))
print(df.groupby("branch")["above_avg"].mean().round(2))

## 6. Tables and Keys

In [ ]:
print(customers.head(3))
print(products.head(3))
print(orders.head(3))

## 7. Joining with merge

In [ ]:
order_lines = orders.merge(products, on="product_id")

order_lines["total"] = order_lines["price"] * order_lines["quantity"]
order_lines[["product", "category", "price", "quantity", "total"]].head(4)

In [ ]:
full = order_lines.merge(customers, on="customer_id")
print(full.shape)

full.groupby(["city", "category"])["total"].sum().unstack().round(0)

## 8. Types of Join

What should happen to a row with no match in the other table? The `how` argument decides:

- `inner` (the default) keeps only rows whose key appears in **both** tables.
- `left` keeps **all** rows of the left table and fills missing details with `NaN`.
- `right` is the mirror image; `outer` keeps all rows from both tables.

Choosing the wrong join silently adds or drops rows, so always check the number of rows before and after.

In [ ]:
inner = orders.merge(customers, on="customer_id", how="inner")
left = orders.merge(customers, on="customer_id", how="left")

print(len(orders), len(inner), len(left))
print(left["name"].isna().sum(), "orders with no matching customer")

| how= | Keeps | Use it when |
| --- | --- | --- |
| `"inner"` | Only keys found in both tables | You only want complete records |
| `"left"` | Every row of the left table | Every order must stay, matched or not |
| `"right"` | Every row of the right table | The mirror of left; rarely needed |
| `"outer"` | Every row of both tables | You want to see all mismatches |

## 9. Finding Rows That Do Not Match

In [ ]:
check = orders.merge(customers, on="customer_id", how="left", indicator=True)
print(check["_merge"].value_counts())

unknown = check.loc[check["_merge"] == "left_only", "customer_id"]
print(sorted(unknown.unique()))

## 10. Stacking Tables with concat

`merge` puts tables **side by side** by matching keys. `pd.concat` puts tables **one under the other**, for example sales files from different months or branches with the same columns.

In [ ]:
jan = orders[orders["order_date"] < "2025-02-01"]
feb = orders[orders["order_date"].between("2025-02-01", "2025-02-28")]

both = pd.concat([jan, feb], ignore_index=True)
print(len(jan), len(feb), len(both))
both.tail(3)

## 11. Wide and Long Tables

The same data can be stored **wide**, with one column per month, or **long**, with one row per month. People read wide tables more easily; charts and `groupby` usually need long tables. `melt` turns wide into long, and `pivot` turns long into wide.

In [ ]:
wide = pd.DataFrame({"branch": ["Campus", "Mall"],
                     "Jan": [690, 410], "Feb": [655, 398], "Mar": [697, 500]})

long = wide.melt(id_vars="branch", var_name="month", value_name="revenue")
print(long)
print(long.pivot(index="branch", columns="month", values="revenue"))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Index confusion | Expecting the group names as a column | Call `.reset_index()` to turn them back into a column. |
| Wrong length | Using `agg` when you needed a value per row | Use `transform` to keep the original number of rows. |
| Rows multiply | Merging on a key that is not unique in either table | Check `df[key].duplicated().sum()` before merging. |
| Rows vanish | Using the default inner join without noticing | Compare row counts; use `how="left"` to keep everything. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Revenue per Item

Find the total revenue of each item and show the top five, highest first.

In [ ]:
# Write your code below

### In-Class Exercise 2: Branch Summary

Build one summary table per branch with the number of orders, the average quantity, the number of different items sold and the total revenue.

In [ ]:
# Write your code below

### In-Class Exercise 3: Product Names on Orders

Merge the orders with the products and show the five most ordered products by total quantity.

In [ ]:
# Write your code below

### In-Class Exercise 4: Check the Join

Count how many orders would be lost by an inner join with the customers table, and list their order IDs.

In [ ]:
# Write your code below

## Summary

- `groupby(col)[value].stat()` summarises every group in one line.
- `agg` builds several statistics at once; `pivot_table` makes a grid.
- Group by a list of columns, or by `df["date"].dt.month`.
- `merge(other, on=key)` joins tables on a shared key.
- `how="inner"` keeps matches only; `how="left"` keeps every left row.
- `concat` stacks tables; `melt` turns wide tables into long ones.

## What's Next

So far every answer was a table of numbers. Next session you will turn tables into charts with Matplotlib and Seaborn, and learn which chart answers which question.